In [ ]:
%pip -q install "timm>=0.9.16" "einops>=0.8" opencv-python-headless kagglehub tqdm matplotlib Pillow

In [ ]:
import os, sys, subprocess
from pathlib import Path



PROJECT_REPO_URL = "https://github.com/asrme2008/CamoDiffusion-Contrastive-Loss.git"
PROJECT_CLONE_DIR = Path("/kaggle/working/camodiffusion-contrastive-project-source")
MODEL_FILE = "diffusion_model.py"
TRAINING_FILE = "camodiffusion_contrastive_training.py"
LOSS_FILE = "contrastive_loss.py"


def find_project_source_root():
    
    roots = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    for base in roots:
        if not base.exists():
            continue
        if (base / MODEL_FILE).is_file() and (base / TRAINING_FILE).is_file() and (base / LOSS_FILE).is_file():
            return base
        
        for dirpath, dirnames, filenames in os.walk(base):
            depth = len(Path(dirpath).relative_to(base).parts)
            dirnames[:] = [d for d in dirnames if d not in {"Imgs", "GT", "Image", "Images", "__MACOSX", ".git", ".ipynb_checkpoints"}]
            if depth > 5:
                dirnames[:] = []
                continue
            if MODEL_FILE in filenames and TRAINING_FILE in filenames and LOSS_FILE in filenames:
                return Path(dirpath)
    return None

PROJECT_ROOT = find_project_source_root()
if PROJECT_ROOT is None:
    if "YOUR_REPO" in PROJECT_REPO_URL:
        raise RuntimeError(
            "Project source files were not found. First upload this project to your own GitHub repository, "
            "check PROJECT_REPO_URL and set it to your own repository .git URL. Do not enter the authors' URL here."
        )
    if PROJECT_CLONE_DIR.exists() and not (PROJECT_CLONE_DIR / MODEL_FILE).exists():
        raise RuntimeError(f"{PROJECT_CLONE_DIR} already exists but does not contain the model source; remove it and rerun.")
    if not PROJECT_CLONE_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(PROJECT_CLONE_DIR)], check=True)
    PROJECT_ROOT = PROJECT_CLONE_DIR

sys.path.insert(0, str(PROJECT_ROOT))
print("Using project source:", PROJECT_ROOT)
print("Model file:", PROJECT_ROOT / MODEL_FILE)
print("Training helper:", PROJECT_ROOT / TRAINING_FILE)

In [ ]:
import json, os, random, time, gc
import torch
from torch.utils.data import DataLoader
from camodiffusion_contrastive_training import (
    CamoDiffusionDataset, SNRScheduler, discover_dataset_pairs, limited_pairs,
    find_attached_file, fit_contrastive_model, push_checkpoint_dataset, seed_everything,
)
from diffusion_model import CamoDiffusionModel


SEED = 42
IMG_SIZE = 352
NUM_EPOCHS = 150
LR = 1e-4
LR_MIN = 1e-6
MICRO_BS = 2            
GRAD_ACCUM = 8          
NUM_WORKERS = 4
USE_FP16 = True
LAMBDA = 0.1            
DELTA = 0.3
W_BOUNDARY = 3.0
MAX_CORRUPTION_ITERATIONS = 50
SAVE_EVERY = 5
KEEP_LAST = 1
MAX_VAL_ITEMS = 256     
MAX_VAL_BATCHES = 16    
MAX_TRAIN_BATCHES = None 
TIME_LIMIT_H = 11.0     
STOP_AFTER_EPOCH = None 
FRESH_START = False
RESULTS_DIR = "/kaggle/working/camodiffusion_no_ofm_contrastive_stage1_352"
DATASET_SLUG = "camodiffusion-no-ofm-contrastive-stage1"
RESUME_HINT = DATASET_SLUG
PUSH_TO_KAGGLE_DATASET = True  
PUSH_EVERY = 5


MANUAL_TRAIN_IMAGE_DIR = None
MANUAL_TRAIN_GT_DIR = None
MANUAL_VAL_IMAGE_DIR = None
MANUAL_VAL_GT_DIR = None


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("WARNING: CUDA is not available. Full training will be very slow on CPU.")

train_pairs, val_pairs, dataset_meta = discover_dataset_pairs(
    "/kaggle/input",
    manual_train_image_dir=MANUAL_TRAIN_IMAGE_DIR,
    manual_train_gt_dir=MANUAL_TRAIN_GT_DIR,
    manual_val_image_dir=MANUAL_VAL_IMAGE_DIR,
    manual_val_gt_dir=MANUAL_VAL_GT_DIR,
)
val_pairs = limited_pairs(val_pairs, MAX_VAL_ITEMS, seed=SEED)
print("Dataset discovery:", json.dumps(dataset_meta, indent=2, default=str))
print("Train pairs:", len(train_pairs), "| validation pairs:", len(val_pairs))
if not train_pairs:
    raise RuntimeError("No training image/mask pairs were found. Check the dataset paths above.")


RESULTS_PATH = Path(RESULTS_DIR)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)
RESUME_CHECKPOINT = None
if not FRESH_START:
    local_ckpt = RESULTS_PATH / "checkpoint_last.pt"
    remote_ckpt = find_attached_file("checkpoint_last.pt", RESUME_HINT)
    RESUME_CHECKPOINT = str(local_ckpt) if local_ckpt.is_file() else remote_ckpt
if RESUME_CHECKPOINT:
    payload = torch.load(RESUME_CHECKPOINT, map_location="cpu", weights_only=False)
    saved_settings = payload.get("settings", {}) if isinstance(payload, dict) else {}
    saved_contrastive = saved_settings.get("contrastive", {}) if isinstance(saved_settings, dict) else {}
    LAMBDA = float(saved_contrastive.get("lambda", LAMBDA))
    DELTA = float(saved_contrastive.get("delta", DELTA))
    W_BOUNDARY = float(saved_contrastive.get("w_boundary", W_BOUNDARY))
    print("Resuming from:", RESUME_CHECKPOINT, "| restored lambda:", LAMBDA)
else:
    print("No prior Stage-1 checkpoint found; starting a new 352x352 run.")

diffusion_scheduler = SNRScheduler(T=1000, device="cpu")
train_ds = CamoDiffusionDataset(
    train_pairs, diffusion_scheduler, img_size=IMG_SIZE, augment=True,
    structure_corruption=True, max_outer_iterations=MAX_CORRUPTION_ITERATIONS,
)
val_ds = CamoDiffusionDataset(
    val_pairs, diffusion_scheduler, img_size=IMG_SIZE, augment=False,
    structure_corruption=False, max_outer_iterations=0,
) if val_pairs else None
train_loader = DataLoader(
    train_ds, batch_size=MICRO_BS, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"), drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds, batch_size=max(1, MICRO_BS), shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"), drop_last=False,
    persistent_workers=(NUM_WORKERS > 0),
) if val_ds is not None else None
print("Train batches:", len(train_loader), "| validation batches:", len(val_loader) if val_loader else 0)

In [ ]:


model = CamoDiffusionModel(img_size=IMG_SIZE, pretrained=(RESUME_CHECKPOINT is None)).to(device)
print("ATCN weight source:", model.weight_source)
print("Parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))

settings = {
    "stage": "stage1_train_352",
    "architecture": "CamoDiffusion without Gated OFM",
    "image_size": IMG_SIZE,
    "num_epochs": NUM_EPOCHS,
    "contrastive": {"lambda": LAMBDA, "delta": DELTA, "w_boundary": W_BOUNDARY,
                    "timestep_weight": "sigmoid(-log_snr/4)"},
    "optimizer": {"name": "AdamW", "lr": LR, "lr_min": LR_MIN},
    "effective_batch": MICRO_BS * GRAD_ACCUM,
    "dataset": dataset_meta,
}
with open(RESULTS_PATH / "contrastive_config.json", "w") as f:
    json.dump(settings, f, indent=2, default=str)


RUN_SANITY_CHECK = False
if RUN_SANITY_CHECK:
    batch = next(iter(train_loader))
    image = batch["image"].to(device); x0 = batch["x0"].to(device)
    xt = batch["xt"].to(device); t = batch["t"].to(device).long().view(-1)
    gt01 = ((x0.float() + 1) * 0.5).clamp(0, 1)
    pred, features = model.forward_with_features(xt, t, image)
    assert pred.shape == x0.shape, (pred.shape, x0.shape)
    assert len(features) == 4, len(features)
    print("Sanity check OK. Prediction:", tuple(pred.shape), "feature maps:", [tuple(x.shape) for x in features])
    del batch, image, x0, xt, t, gt01, pred, features
    model.zero_grad(set_to_none=True); gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

on_checkpoint = None
if PUSH_TO_KAGGLE_DATASET:
    on_checkpoint = lambda folder, epoch: print(push_checkpoint_dataset(
        folder, DATASET_SLUG, f"No-OFM CamoDiffusion Stage 1 (352px), checkpoint epoch {epoch}; contrastive lambda={LAMBDA}"
    ))

model, history = fit_contrastive_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    diffusion_scheduler=diffusion_scheduler,
    results_dir=RESULTS_DIR,
    epochs=NUM_EPOCHS,
    lr=LR,
    lr_min=LR_MIN,
    contrastive_lambda=LAMBDA,
    delta=DELTA,
    w_boundary=W_BOUNDARY,
    grad_accum=GRAD_ACCUM,
    fp16=USE_FP16,
    num_workers=NUM_WORKERS,
    save_every=SAVE_EVERY,
    keep_last=KEEP_LAST,
    time_limit_hours=TIME_LIMIT_H,
    stop_after_epoch=STOP_AFTER_EPOCH,
    max_train_batches=MAX_TRAIN_BATCHES,
    max_val_batches=MAX_VAL_BATCHES,
    resume_checkpoint=RESUME_CHECKPOINT,
    init_model_checkpoint=None,
    on_checkpoint=on_checkpoint,
    push_every=PUSH_EVERY,
    settings=settings,
)
print("Stage 1 finished/stopped. Files saved in:", RESULTS_DIR)
print("Latest full checkpoint:", RESULTS_PATH / "checkpoint_last.pt")
print("Best model weights:", RESULTS_PATH / "model-best.pt")